# Hunyuan3D-2 MV: Colab L4 GPU Headless API Pipeline

支援全自動透過 Google Colab CLI 無人值守執行 2D 三視圖轉 3D GLB 模型生成。

In [ ]:
import os, shutil, subprocess, sys, re
from pathlib import Path

def run_cmd(cmd, cwd=None):
    res = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"❌ 指令失敗: {' '.join(cmd)}", flush=True)
        print("--- STDERR ---", flush=True)
        print(res.stderr[-3000:], flush=True)
        print("--- STDOUT ---", flush=True)
        print(res.stdout[-3000:], flush=True)
        raise RuntimeError(f"執行失敗: {res.stderr[-500:]}")
    return res.stdout

SETUP_MARKER = Path("/content/.hy3d_setup_done")
COMFY_DIR = Path("/content/ComfyUI")

# 1. 確保 ComfyUI 本體完整就緒
if not (COMFY_DIR / "main.py").exists():
    print("正在下載 ComfyUI 核心庫...")
    if COMFY_DIR.exists():
        shutil.rmtree(str(COMFY_DIR), ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)])

# 2. 確保 custom_nodes 目錄與自訂節點完整就緒
custom_nodes = COMFY_DIR / "custom_nodes"
custom_nodes.mkdir(parents=True, exist_ok=True)

hy3d_dir = custom_nodes / "ComfyUI-Hunyuan3DWrapper"
essentials_dir = custom_nodes / "ComfyUI_essentials"

if not (hy3d_dir / "requirements.txt").exists():
    print("下載 ComfyUI-Hunyuan3DWrapper...")
    if hy3d_dir.exists():
        shutil.rmtree(str(hy3d_dir), ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "https://github.com/kijai/ComfyUI-Hunyuan3DWrapper.git", str(hy3d_dir)])

if not essentials_dir.exists():
    print("下載 ComfyUI_essentials...")
    run_cmd(["git", "clone", "--depth", "1", "https://github.com/cubiq/ComfyUI_essentials.git", str(essentials_dir)])

# 3. 安裝依賴與編譯 CUDA 渲染核心
if not SETUP_MARKER.exists():
    print("安裝 Python 模組依賴...", flush=True)
    run_cmd([sys.executable, "-m", "pip", "install", "--break-system-packages", "-r", str(COMFY_DIR / "requirements.txt")])
    run_cmd([sys.executable, "-m", "pip", "install", "--break-system-packages", "-r", str(hy3d_dir / "requirements.txt")])
    run_cmd([sys.executable, "-m", "pip", "install", "--break-system-packages", "transformers==4.46.0", "accelerate==1.1.1", "diffusers==0.30.0", "rembg[gpu]", "trimesh", "pymeshlab", "xatlas", "huggingface_hub"])

    # 防禦性補丁：確保 pipelines.py 支援 DINOv2 權重鍵名映射且清空所有 meta tensor
    pipe_py = hy3d_dir / "hy3dgen" / "shapegen" / "pipelines.py"
    if pipe_py.exists():
        pipe_code = pipe_py.read_text(encoding="utf-8")
        if "🔥 Conditioner 權重加載" not in pipe_code:
            cond_replacement = '''        if \x27conditioner\x27 in new_sd:
            c_sd = new_sd[\x27conditioner\x27]
            loaded_count = 0
            cond_params = list(conditioner.named_parameters())
            for name, param in cond_params:
                val = c_sd.get(name)
                if val is None:
                    alias = name.replace(\x27.attention.q_proj.\x27, \x27.attention.attention.query.\x27) \\
                                .replace(\x27.attention.k_proj.\x27, \x27.attention.attention.key.\x27) \\
                                .replace(\x27.attention.v_proj.\x27, \x27.attention.attention.value.\x27) \\
                                .replace(\x27.attention.o_proj.\x27, \x27.attention.output.dense.\x27)
                    val = c_sd.get(alias)
                if val is None:
                    alias2 = name.replace(\x27.attention.attention.query.\x27, \x27.attention.q_proj.\x27) \\
                                 .replace(\x27.attention.attention.key.\x27, \x27.attention.k_proj.\x27) \\
                                 .replace(\x27.attention.attention.value.\x27, \x27.attention.v_proj.\x27) \\
                                 .replace(\x27.attention.output.dense.\x27, \x27.attention.o_proj.\x27)
                    val = c_sd.get(alias2)
                if val is not None:
                    set_module_tensor_to_device(conditioner, name, device=offload_device, dtype=dtype, value=val)
                    loaded_count += 1
                else:
                    set_module_tensor_to_device(conditioner, name, device=offload_device, dtype=dtype, value=torch.zeros(param.shape, dtype=dtype))
            print(f"🔥 Conditioner 權重加載: 成功加載 {loaded_count}/{len(cond_params)} 個參數！", flush=True)

        for m in [conditioner, model, vae]:
            for p_name, param in m.named_parameters():
                if getattr(param, \x27is_meta\x27, False):
                    set_module_tensor_to_device(m, p_name, device=offload_device, dtype=dtype, value=torch.zeros(param.shape, dtype=dtype))
            for b_name, buf in m.named_buffers():
                if getattr(buf, \x27is_meta\x27, False):
                    set_module_tensor_to_device(m, b_name, device=offload_device, dtype=buf.dtype, value=torch.zeros(buf.shape, dtype=buf.dtype))

        image_processor = instantiate_from_config(config[\x27image_processor\x27])'''

            cond_regex = re.compile(r"        if \x27conditioner\x27 in new_sd:.*?image_processor = instantiate_from_config\(config\[\x27image_processor\x27\]\)", re.DOTALL)
            if cond_regex.search(pipe_code):
                pipe_code = cond_regex.sub(cond_replacement, pipe_code, count=1)
                pipe_py.write_text(pipe_code, encoding="utf-8")
                print("✅ 已對 pipelines.py 注入完整 DINOv2 權重映射與 Meta Tensor 清理補丁！")

    # 防禦性補丁 2：確保 HunyuanPaintPipeline 支援 RGBA 自動白底合成轉 RGB (避免 VAE 4 通道報錯)
    paint_pipe_py = hy3d_dir / "hy3dgen" / "texgen" / "hunyuanpaint" / "pipeline.py"
    if paint_pipe_py.exists():
        p_code = paint_pipe_py.read_text(encoding="utf-8")
        p_target = "posterior = self.vae.encode(images.to(dtype)).latent_dist"
        p_safe = """if images.shape[1] == 4:
            alpha = images[:, 3:4]
            images = images[:, :3] * alpha + (1.0 - alpha)
        posterior = self.vae.encode(images.to(dtype)).latent_dist"""
        if p_target in p_code and "images.shape[1] == 4" not in p_code:
            p_code = p_code.replace(p_target, p_safe)
            paint_pipe_py.write_text(p_code, encoding="utf-8")
            print("✅ 已對 hunyuanpaint/pipeline.py 注入 RGBA->RGB 兼容補丁！")

    # 防禦性補丁 3：確保 nodes.py Hy3DSampleMultiView 支援 4 通道 ref_image
    nodes_py = hy3d_dir / "nodes.py"
    if nodes_py.exists():
        n_code = nodes_py.read_text(encoding="utf-8")
        n_target = "multiview_images = pipeline("
        n_safe = """if hasattr(ref_image, \x27shape\x27) and ref_image.shape[-1] == 4:
            ref_image = ref_image[..., :3] * ref_image[..., 3:4] + (1.0 - ref_image[..., 3:4])
        multiview_images = pipeline("""
        if n_target in n_code and "ref_image.shape[-1] == 4" not in n_code:
            n_code = n_code.replace(n_target, n_safe, 1)
            nodes_py.write_text(n_code, encoding="utf-8")
            print("✅ 已對 nodes.py 注入 ref_image RGBA->RGB 兼容補丁！")
    rasterizer_dir = hy3d_dir / "hy3dgen" / "texgen" / "custom_rasterizer"
    if rasterizer_dir.exists():
        print("編譯 custom_rasterizer CUDA 模組...")
        run_cmd([sys.executable, "setup.py", "install"], cwd=str(rasterizer_dir))
    SETUP_MARKER.touch()
    print("✅ 核心軟體環境已安裝就緒！")
else:
    print("✅ 重用已配置的 Hunyuan3D-2 軟體環境。")

# 4. 下載 Hunyuan3D-2 MV 核心 DiT 權重 (約 4.9 GB)
diff_dir = COMFY_DIR / "models" / "diffusion_models"
diff_dir.mkdir(parents=True, exist_ok=True)
model_file = diff_dir / "hunyuan3d-dit-v2-mv-fp16.safetensors"

if not model_file.exists() or model_file.stat().st_size < 4 * 1024 * 1024 * 1024:
    print("正在自 HuggingFace 下載 hunyuan3d-dit-v2-mv-fp16.safetensors (約 4.9GB)...", flush=True)
    from huggingface_hub import hf_hub_download
    dl_path = hf_hub_download(
        repo_id="tencent/Hunyuan3D-2mv",
        filename="hunyuan3d-dit-v2-mv/model.fp16.safetensors",
        local_dir="/content/hf_temp",
        local_dir_use_symlinks=False
    )
    shutil.move(dl_path, str(model_file))
    print(f"✅ 模型權重下載成功！大小: {model_file.stat().st_size / (1024*1024):.2f} MB")
else:
    print(f"✅ 核心權重已存在 ({model_file.stat().st_size / (1024*1024):.2f} MB)，跳過下載。")


In [ ]:
import json, time, requests, subprocess, sys
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
PORT = 8188
BASE_URL = f"http://127.0.0.1:{PORT}"
LOG_PATH = Path("/content/comfyui.log")

try:
    resp = requests.get(f"{BASE_URL}/system_stats", timeout=2)
    resp.raise_for_status()
    print("✅ 使用背景現存的 ComfyUI 服務")
except Exception:
    print("正在啟動 ComfyUI 服務...")
    log_f = open(LOG_PATH, "a")
    cmd = [sys.executable, str(COMFY_DIR / "main.py"), "--listen", "127.0.0.1", "--port", str(PORT), "--highvram", "--disable-auto-launch"]
    proc = subprocess.Popen(cmd, cwd=str(COMFY_DIR), stdout=log_f, stderr=subprocess.STDOUT)
    for _ in range(120):
        if proc.poll() is not None:
            raise RuntimeError(f"ComfyUI 啟動失敗；查看 {LOG_PATH}\n" + LOG_PATH.read_text()[-3000:])
        try:
            requests.get(f"{BASE_URL}/system_stats", timeout=2).raise_for_status()
            print("✅ ComfyUI 啟動就緒！")
            break
        except Exception:
            time.sleep(2)
    else:
        raise TimeoutError(f"ComfyUI 啟動逾時；查看 {LOG_PATH}")

In [ ]:
import os, shutil, json, time, uuid, requests
from pathlib import Path

PORT = 8188
BASE_URL = f"http://127.0.0.1:{PORT}"
LOG_PATH = Path("/content/comfyui.log")
COMFY_INPUT = Path("/content/ComfyUI/input")
COMFY_OUTPUT = Path("/content/ComfyUI/output")
COMFY_INPUT.mkdir(parents=True, exist_ok=True)
COMFY_OUTPUT.mkdir(parents=True, exist_ok=True)

front_img = os.environ.get("HY3D_FRONT_IMAGE", "/content/input_front.png")
back_img = os.environ.get("HY3D_BACK_IMAGE", "/content/input_back.png")
side_img = os.environ.get("HY3D_SIDE_IMAGE", "/content/input_side.png")
target_faces = int(os.environ.get("HY3D_TARGET_FACES", "15000"))
output_path = Path(os.environ.get("HY3D_OUTPUT_PATH", "/content/output_model.glb"))
output_path.parent.mkdir(parents=True, exist_ok=True)

assert os.path.exists(front_img), f"找不到正面圖片: {front_img}"
assert os.path.exists(back_img), f"找不到背面圖片: {back_img}"
assert os.path.exists(side_img), f"找不到側面圖片: {side_img}"

job_id = uuid.uuid4().hex[:8]
fname_front = f"front_{job_id}.png"
fname_back = f"back_{job_id}.png"
fname_side = f"side_{job_id}.png"

shutil.copy2(front_img, COMFY_INPUT / fname_front)
shutil.copy2(back_img, COMFY_INPUT / fname_back)
shutil.copy2(side_img, COMFY_INPUT / fname_side)
print(f"已複製輸入圖片至 ComfyUI input 目錄 (Job: {job_id})")

prefix = f"hy3d_out_{job_id}"
workflow_file = Path("/content/workflow_api.json")
assert workflow_file.exists(), f"找不到工作流檔: {workflow_file}"
workflow = json.loads(workflow_file.read_text(encoding="utf-8"))

workflow["10"]["inputs"]["image"] = fname_front
workflow["11"]["inputs"]["image"] = fname_back
workflow["12"]["inputs"]["image"] = fname_side
workflow["32"]["inputs"]["max_facenum"] = target_faces
workflow["40"]["inputs"]["filename_prefix"] = prefix

client_id = str(uuid.uuid4())
prompt_res = requests.post(f"{BASE_URL}/prompt", json={"prompt": workflow, "client_id": client_id}, timeout=30).json()
prompt_id = prompt_res.get("prompt_id")
if not prompt_id:
    raise RuntimeError(f"ComfyUI 驗證或提交任務失敗: {json.dumps(prompt_res, indent=2)}")
print(f"🚀 已成功提交推論任務，Prompt ID: {prompt_id}")

# 輪詢等待完成 (上限 30 分鐘；容忍 xatlas C++ 運算期間短暫阻塞 HTTP 回應)
start_time = time.time()
last_hist = None
while time.time() - start_time < 1800:
    try:
        hist_resp = requests.get(f"{BASE_URL}/history/{prompt_id}", timeout=60)
        if hist_resp.status_code == 200:
            data = hist_resp.json()
            if prompt_id in data:
                last_hist = data[prompt_id]
                print("🎉 ComfyUI 流程回傳！檢查狀態與結果...")
                break
    except requests.exceptions.RequestException as req_err:
        print(f"⏳ ComfyUI 正在密集運算中，繼續等待... ({type(req_err).__name__})", flush=True)
    time.sleep(5)

if not last_hist:
    raise TimeoutError("推論執行逾時")

status_info = last_hist.get("status", {})
outputs_info = last_hist.get("outputs", {})
print("執行狀態:", json.dumps(status_info, ensure_ascii=False, indent=2))
print("節點輸出:", json.dumps(outputs_info, ensure_ascii=False, indent=2))

if not status_info.get("completed", False) or status_info.get("status_str") == "error":
    print("--- ComfyUI Log Tail ---")
    if LOG_PATH.exists():
        print(LOG_PATH.read_text()[-4000:])
    raise RuntimeError(f"ComfyUI 推論中斷或報錯: {status_info.get('messages')}")

# 遞迴搜尋產出的 .glb 檔案 (包含 output 及其所有子目錄與 /content)
found_glb = None
for p in COMFY_OUTPUT.rglob("*.glb"):
    if prefix in p.name:
        found_glb = p
        break
if not found_glb:
    glbs = sorted(list(COMFY_OUTPUT.rglob("*.glb")), key=os.path.getmtime, reverse=True)
    if glbs:
        found_glb = glbs[0]
if not found_glb:
    all_glbs = sorted(list(Path("/content").rglob("*.glb")), key=os.path.getmtime, reverse=True)
    if all_glbs:
        found_glb = all_glbs[0]

if not found_glb or not found_glb.exists():
    print("output 目錄樹:", list(COMFY_OUTPUT.rglob("*")))
    if LOG_PATH.exists():
        print("--- ComfyUI Log Tail ---")
        print(LOG_PATH.read_text()[-4000:])
    raise AssertionError(f"未找到產出的 .glb 檔案！")

if LOG_PATH.exists():
    print("--- ComfyUI Log Summary ---")
    for l in LOG_PATH.read_text().splitlines():
        if any(w in l for w in ["Conditioner", "Hy3D", "Prompt executed in", "Error", "Exception"]):
            print(f"[ComfyUI Log] {l}")

shutil.copy2(found_glb, output_path)
print(f"✅ 模型成功導出至: {output_path} (大小: {output_path.stat().st_size / (1024*1024):.2f} MB)")
